<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/testing_trade_thesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade yfinance
!pip install  --upgrade pandas_ta
!pip install pandas-ta
!pip install ta
!pip install scipy==1.16.2

  Using cached scipy-1.16.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (62 kB)
Using cached scipy-1.16.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (35.7 MB)
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [2]:
import yfinance as yf
print(yf.__version__)
import pandas as pd
import ta
import numpy as np
import requests
from datetime import datetime, timedelta
from scipy.stats import linregress
import time
import random
random.seed(42)
print("Libraries Installed!")

1.2.1
Libraries Installed!


In [3]:
# Function to compute ATR (Average True Range)
def compute_atr(df, period=10):
  try:
    df['High-Low'] = df['High'] - df['Low']
    df['High-Close'] = abs(df['High'] - df['Close'].shift(1))
    df['Low-Close'] = abs(df['Low'] - df['Close'].shift(1))
    df['TR'] = df[['High-Low', 'High-Close', 'Low-Close']].max(axis=1)
    return df['TR'].rolling(window=period).mean()
  except Exception as e:
      print("Something went wrong whilecomputing the ATR", e)

# Function to fetch historical weekly data
def rolling_regression_slope(series, window=10):
    """Rolling linear regression slope (price units per bar)."""
    def calc_slope(y):
        if len(y) < 2:
            return np.nan
        x = np.arange(len(y))
        slope, _, _, _, _ = linregress(x, y)
        return slope
    return series.rolling(window).apply(calc_slope, raw=False)

In [9]:
df = yf.download('CSGP', period="1y", interval="1d",auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
  df.columns = df.columns.get_level_values(0)  # keep only first level

df['20_day_SMA'] = df['Close'].rolling(window=20).mean()
df['50_day_avg_volume'] = df['Volume'].rolling(window=50).mean()
df['8_day_EMA'] = df['Close'].ewm(span=8, adjust=False).mean()
df['15_day_EMA'] = df['Close'].ewm(span=15, adjust=False).mean()
df['21_day_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
df['26_day_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
df['5_day_SMA'] = df['Close'].rolling(window=5).mean()
df['50_day_SMA'] = df['Close'].rolling(window=50).mean()
df['8_day_EMA'] = df['Close'].ewm(span=8, adjust=False).mean()

# Slope for 5-day SMA (short-term trend)
df['slope5_raw'] = rolling_regression_slope(df['5_day_SMA'], window=10)
df['slope5_pct_per_day'] = df['slope5_raw'] / df['5_day_SMA']  # fractional change per day
df['slope5_annualized_pct'] = df['slope5_pct_per_day'] * 252 * 100  # % per year
# Slope for 50-day SMA (intermediate-term trend)
df['slope50_raw'] = rolling_regression_slope(df['50_day_SMA'], window=10)
df['slope50_pct_per_day'] = df['slope50_raw'] / df['50_day_SMA']  # fractional change per day
df['slope50_annualized_pct'] = df['slope50_pct_per_day'] * 252 * 100  # % per year
df['100_day_SMA'] = df['Close'].rolling(window=100).mean()
df['200_day_SMA'] = df['Close'].rolling(window=200).mean()
df['ATR'] = compute_atr(df, 10)
df["8EMA_minus_ATR"] = df["8_day_EMA"] - 0.45* df["ATR"]
df["8EMA_minus_ATRL"] = df["8_day_EMA"] - 1* df["ATR"]
# 1️⃣ Yesterday touched or pierced 8 EMA
df['prior_touch_8ema'] = df['Low'].shift(1) <= df['8_day_EMA'].shift(1)
# 2️⃣ Today closes above 8 EMA
df['close_above_8ema'] = df['Close'] > df['8_day_EMA']
# 3️⃣ Today closes above yesterday’s close (price rising)
df['close_above_prev_close'] = df['Close'] > df['Close'].shift(1)
# 4️⃣ Strong confirmation: Break previous high
df['break_prev_high'] = df['Close'] > df['High'].shift(1)
# 5️⃣ 8 EMA slope positive (trend filter)
df['ema8_rising'] = df['8_day_EMA'] > df['8_day_EMA'].shift(1)
# EMA 8 slope
df['ema8_slope'] = df['8_day_EMA'] - df['8_day_EMA'].shift(1)
# EMA 8 slope previous
df['ema8_slope_prev'] = df['ema8_slope'].shift(1)
# EMA 8 acceleration
df['ema8_accel'] = df['ema8_slope'] - df['ema8_slope_prev']
# EMA 8 slope direction (1 = up, 0 = down)
df['ema8_dir'] = (df['ema8_slope'] > 0).astype(int)
# Count direction changes over last 5 days
df['ema8_direction_changes'] = (
      df['ema8_dir']
      .diff()
      .abs()
      .rolling(5)
      .sum()
      )
# 1. Higher Timeframe Bearish Bias (Monthly + Weekly already confirmed)
trend_short = df['slope50_raw'] < 0

# 2. Avoid strong counter-trend moves and chop
df['daily_return'] = df['Close'].pct_change()
avoid_strong_up = df['daily_return'] > 0.035          # Strong bullish day
avoid_chop = df['ema8_direction_changes'] >= 3

# 3. EMA 8 Price Action
df['touched_ema8']      = df['High'] >= df['8_day_EMA'] * 0.995
df['closed_below_ema8'] = df['Close'] < df['8_day_EMA']
df['bearish_candle']    = df['Close'] < df['Open']

# Strong upper wick rejection
df['upper_wick_ratio']  = (df['High'] - df['Close']) / (df['High'] - df['Low'] + 0.0001)
df['strong_upper_wick'] = df['upper_wick_ratio'] > 0.60

# 4. Advanced Bearish Patterns
df['bearish_engulfing'] = (
    (df['Close'] < df['Open']) &
    (df['Open'] > df['Close'].shift(1)) &
    (df['Close'] < df['Close'].shift(1))
)

df['failed_break_ema8'] = (
    (df['Close'].shift(1) > df['8_day_EMA'].shift(1)) &
    (df['Close'] < df['8_day_EMA'])
)

df['near_50sma'] = df['High'] >= df['50_day_SMA'] * 0.99

# 5. A and A+ Setups
df['A_setup'] = (
    df['touched_ema8'] &
    df['closed_below_ema8'] &
    df['bearish_candle'] &
    df['strong_upper_wick']
)

df['A_plus_setup'] = (
    df['failed_break_ema8'] &
    #(df['bearish_engulfing'] | df['strong_upper_wick']) &
    df['bearish_engulfing'] &
    df['closed_below_ema8'] &
    (df['near_50sma'] | df['strong_upper_wick'])
)

# 6. Entry Trigger (Momentum)
df['entry_trigger'] = df['Low'] < df['Low'].shift(1)

# ====================== FINAL SHORT SIGNAL ======================
df['short_signal'] = (
    trend_short &                          # Higher TF bearish bias
    (~avoid_strong_up) &
    (~avoid_chop) &
    (df['A_setup'] | df['A_plus_setup']) &
    df['entry_trigger']
)

#df['short_signal'] = (
    #trend_short &
    #(~avoid_strong_up) &
    #(~avoid_chop) &
    #(df['A_setup'].shift(1) | df['A_plus_setup'].shift(1)) &
    #df['entry_trigger']
#)

# Signal Strength Labeling
df['signal_strength'] = 'None'
df.loc[df['short_signal'] & df['A_plus_setup'], 'signal_strength'] = 'A+'
df.loc[df['short_signal'] & df['A_setup'] & ~df['A_plus_setup'], 'signal_strength'] = 'A'

df.tail(15)

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume,20_day_SMA,50_day_avg_volume,8_day_EMA,15_day_EMA,21_day_EMA,...,upper_wick_ratio,strong_upper_wick,bearish_engulfing,failed_break_ema8,near_50sma,A_setup,A_plus_setup,entry_trigger,short_signal,signal_strength
Date,,,,,,,,,,,,,,,,,,,,,
2026-03-20,42.900002,43.270000,41.959999,42.240002,15182900,45.6025,7255986.0,43.875659,45.009146,46.074726,...,0.282420,False,False,False,False,False,False,False,False,None
2026-03-23,42.910000,43.750000,41.790001,43.669998,7209300,45.3545,7233414.0,43.661068,44.746753,45.787024,...,0.428550,False,False,False,False,False,False,True,False,None
2026-03-24,41.459999,42.520000,40.779999,42.320000,4556800,44.9700,7170738.0,43.171942,44.335908,45.393658,...,0.609161,True,False,False,False,False,False,True,False,None
2026-03-25,41.410000,42.119999,40.330002,41.840000,5753000,44.8015,7108134.0,42.780399,43.970170,45.031507,...,0.396626,False,True,False,False,False,False,True,False,None
2026-03-26,41.299999,42.709999,40.950001,40.950001,4984300,44.6165,6953836.0,42.451421,43.636398,44.692279,...,0.801092,True,False,False,False,False,False,False,False,None
2026-03-27,39.770000,41.430000,39.180000,41.070000,6295200,44.3735,6915676.0,41.855550,43.153099,44.244799,...,0.737745,True,False,False,False,False,False,True,False,None
2026-03-30,40.880001,41.070000,40.040001,40.160000,6667800,44.1590,6939602.0,41.638761,42.868962,43.938909,...,0.184447,False,False,False,False,False,False,False,False,None
2026-03-31,40.340000,41.630001,39.570000,41.500000,5481900,43.8370,6896782.0,41.350148,42.552841,43.611735,...,0.626183,True,True,False,False,True,False,True,True,A
2026-04-01,39.630001,40.340000,38.689999,40.340000,5478000,43.4385,6891914.0,40.967893,42.187486,43.249759,...,0.430276,False,False,False,False,False,False,True,False,None


In [5]:
df = yf.download('lrcx', period="1y", interval="1d",auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
  df.columns = df.columns.get_level_values(0)  # keep only first level

df['20_day_SMA'] = df['Close'].rolling(window=20).mean()
df['50_day_avg_volume'] = df['Volume'].rolling(window=50).mean()
df['8_day_EMA'] = df['Close'].ewm(span=8, adjust=False).mean()
df['15_day_EMA'] = df['Close'].ewm(span=15, adjust=False).mean()
df['21_day_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
df['26_day_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
df['5_day_SMA'] = df['Close'].rolling(window=5).mean()
df['50_day_SMA'] = df['Close'].rolling(window=50).mean()
df['8_day_EMA'] = df['Close'].ewm(span=8, adjust=False).mean()

# Slope for 5-day SMA (short-term trend)
df['slope5_raw'] = rolling_regression_slope(df['5_day_SMA'], window=10)
df['slope5_pct_per_day'] = df['slope5_raw'] / df['5_day_SMA']  # fractional change per day
df['slope5_annualized_pct'] = df['slope5_pct_per_day'] * 252 * 100  # % per year
# Slope for 50-day SMA (intermediate-term trend)
df['slope50_raw'] = rolling_regression_slope(df['50_day_SMA'], window=10)
df['slope50_pct_per_day'] = df['slope50_raw'] / df['50_day_SMA']  # fractional change per day
df['slope50_annualized_pct'] = df['slope50_pct_per_day'] * 252 * 100  # % per year
df['100_day_SMA'] = df['Close'].rolling(window=100).mean()
df['200_day_SMA'] = df['Close'].rolling(window=200).mean()
df['ATR'] = compute_atr(df, 10)
df["8EMA_minus_ATR"] = df["8_day_EMA"] - 0.45* df["ATR"]
df["8EMA_minus_ATRL"] = df["8_day_EMA"] - 1* df["ATR"]
# 1️⃣ Yesterday touched or pierced 8 EMA
df['prior_touch_8ema'] = df['Low'].shift(1) <= df['8_day_EMA'].shift(1)
# 2️⃣ Today closes above 8 EMA
df['close_above_8ema'] = df['Close'] > df['8_day_EMA']
# 3️⃣ Today closes above yesterday’s close (price rising)
df['close_above_prev_close'] = df['Close'] > df['Close'].shift(1)
# 4️⃣ Strong confirmation: Break previous high
df['break_prev_high'] = df['Close'] > df['High'].shift(1)
# 5️⃣ 8 EMA slope positive (trend filter)
df['ema8_rising'] = df['8_day_EMA'] > df['8_day_EMA'].shift(1)
# EMA 8 slope
df['ema8_slope'] = df['8_day_EMA'] - df['8_day_EMA'].shift(1)
# EMA 8 slope previous
df['ema8_slope_prev'] = df['ema8_slope'].shift(1)
# EMA 8 acceleration
df['ema8_accel'] = df['ema8_slope'] - df['ema8_slope_prev']
# EMA 8 slope direction (1 = up, 0 = down)
df['ema8_dir'] = (df['ema8_slope'] > 0).astype(int)
# Count direction changes over last 5 days
df['ema8_direction_changes'] = (
      df['ema8_dir']
      .diff()
      .abs()
      .rolling(5)
      .sum()
      )
# 1. Higher Timeframe Bearish Bias (Monthly + Weekly already confirmed)
trend_long = df['slope50_raw'] > 0

# 2. Avoid strong down days and chop
df['daily_return'] = df['Close'].pct_change()
avoid_strong_down = df['daily_return'] < -0.035
avoid_chop = df['ema8_direction_changes'] >= 3

# --- Common Conditions ---
df['closed_above_ema8'] = df['Close'] > df['8_day_EMA']
df['bullish_candle'] = df['Close'] > df['Open']
df['higher_high'] = df['High'] > df['High'].shift(1)

# --- Pullback Setups (A / A+) ---
# 3. EMA 8 Price Action
df['touched_ema8']      = df['Low'] <= df['8_day_EMA'] * 1.005
df['strong_lower_wick'] = ((df['Close'] - df['Low']) / (df['High'] - df['Low'] + 0.0001)) > 0.58
#df['lower_wick_ratio']  = (df['Close'] - df['Low']) / (df['High'] - df['Low'] + 0.0001)
#df['strong_lower_wick'] = df['lower_wick_ratio'] > 0.60

# 4. Advanced Bullish Patterns
df['bullish_engulfing'] = (
    (df['Close'] > df['Open']) &
    (df['Open'] < df['Close'].shift(1)) &
    (df['Close'] > df['Close'].shift(1))
)

df['failed_break_below'] = (
    ((df['Close'].shift(1) < df['8_day_EMA'].shift(1)) |
     (df['Close'].shift(2) < df['8_day_EMA'].shift(2))) &
    df['closed_above_ema8']
)

# 5. A and A+ Setups
df['A_setup_long'] = (
    df['touched_ema8'] &
    df['closed_above_ema8'] &
    df['bullish_candle'] &
    df['strong_lower_wick']
)

df['A_plus_setup_long'] = (
    df['failed_break_below'] &
    (df['bullish_engulfing'] | df['strong_lower_wick']) &
    df['closed_above_ema8']
)

# --- 2. Trend Continuation / Momentum Trades ---
df['trend_continuation'] = (
    (df['Close'] > df['8_day_EMA']) &                    # Already above EMA
    (df['Close'].shift(1) > df['8_day_EMA'].shift(1)) &  # Was already above yesterday
    df['higher_high'] &                                   # Making higher highs
    (df['ema8_slope'] > 0) &                             # EMA sloping up
    (df['daily_return'] > 0.005)                          # Decent green candle
)
## Final Long Signal
df['long_signal'] = (
    trend_long &
    (~avoid_strong_down) &
    (~avoid_chop) &
    (
        (df['A_setup_long'] | df['A_plus_setup_long']) |   # Pullback setups
        df['trend_continuation']                           # Trend continuation trades
    ) &
    df['higher_high']
)
# Signal Strength Labeling
df['signal_type'] = 'None'
df.loc[df['long_signal'] & (df['A_setup_long'] | df['A_plus_setup_long']), 'signal_type'] = 'Pullback (A/A+)'
df.loc[df['long_signal'] & df['trend_continuation'], 'signal_type'] = 'Trend Continuation'
df.loc[df['long_signal'] & (df['A_setup_long'] | df['A_plus_setup_long']) & df['trend_continuation'], 'signal_type'] = 'Hybrid'

# Strength
df['signal_strength'] = 'None'
df.loc[df['long_signal'] & df['A_plus_setup_long'], 'signal_strength'] = 'A+'
df.loc[df['long_signal'] & df['A_setup_long'] & ~df['A_plus_setup_long'], 'signal_strength'] = 'A'
df.loc[df['long_signal'] & df['trend_continuation'], 'signal_strength'] = 'Trend'

df.tail(15)

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume,20_day_SMA,50_day_avg_volume,8_day_EMA,15_day_EMA,21_day_EMA,...,touched_ema8,strong_lower_wick,bullish_engulfing,failed_break_below,A_setup_long,A_plus_setup_long,trend_continuation,long_signal,signal_type,signal_strength
Date,,,,,,,,,,,,,,,,,,,,,
2026-03-20,228.360001,236.839996,222.050003,234.320007,26176400,224.605842,11979588.0,224.209492,223.459824,223.690375,...,True,False,False,False,False,False,False,False,None,None
2026-03-23,233.309998,239.520004,227.070007,230.389999,9506200,224.172337,11891140.0,226.231826,224.691096,224.564886,...,True,False,False,False,False,False,True,True,Trend Continuation,Trend
2026-03-24,238.839996,241.369995,230.160004,230.160004,7388000,223.916451,11703728.0,229.033642,226.459708,225.862624,...,True,True,True,False,True,False,True,True,Hybrid,Trend
2026-03-25,233.449997,237.100006,227.360001,236.149994,8330600,223.129879,11655358.0,230.015054,227.333494,226.552385,...,True,True,False,False,False,False,False,False,None,None
2026-03-26,211.619995,226.889999,211.380005,225.740005,13460500,221.771683,11669828.0,225.927263,225.369307,225.194895,...,True,False,False,False,False,False,False,False,None,None
2026-03-27,211.410004,217.000000,209.000000,209.669998,8503300,220.661678,11476150.0,222.701205,223.624394,223.941723,...,True,False,False,False,False,False,False,False,None,None
2026-03-30,199.929993,216.289993,198.600006,214.559998,11921000,219.121999,11409574.0,217.640936,220.662594,221.758838,...,True,False,False,False,False,False,False,False,None,None
2026-03-31,213.660004,213.839996,203.009995,206.000000,11725000,218.954499,11356220.0,216.756284,219.787270,221.022581,...,True,True,False,False,False,False,False,False,None,None
2026-04-01,222.009995,225.500000,215.000000,215.449997,10697600,218.905499,11338138.0,217.923775,220.065111,221.112346,...,True,True,False,True,True,True,False,True,Pullback (A/A+),A+
